In [1]:
from pathlib import Path
import os

# Find project root by folder name
cwd = Path(os.getcwd())

while cwd.name != "dfsc":
    cwd = cwd.parent

os.chdir(cwd)

print("Working directory set to:", os.getcwd())

Working directory set to: c:\Users\Devanshi\dfsc


In [2]:
# ============================================================
# notebooks/02_preprocessing_feature_engineering.ipynb
# ============================================================

import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from src.feature_engineering import build_full_feature_set, chronological_split
from src.config import TRAIN_END_DATE

# Load the EDA-enriched dataset (much faster than re-loading from CSV)
df = pd.read_parquet("data/processed/eda_enriched.parquet")
print(f"Loaded: {df.shape}")

# Build all features
df_features = build_full_feature_set(df)

# Split chronologically
train_df, test_df = chronological_split(df_features, TRAIN_END_DATE)

# Log-transform Sales for XGBoost (store original for evaluation)
# log1p(x) = log(1+x) — handles zero values safely
train_df["Sales_log"] = np.log1p(train_df["Sales"])
test_df["Sales_log"]  = np.log1p(test_df["Sales"])

# Save both splits
train_df.to_parquet("data/processed/train_features.parquet", index=False)
test_df.to_parquet("data/processed/test_features.parquet",  index=False)

print(f"\nFeature set columns ({len(df_features.columns)}):")
print(list(df_features.columns))
print("\nPreprocessing complete. Ready for modelling.")

Loaded: (844392, 24)
Adding date features ...
Adding German holiday features ...
Adding Promo2 active flag ...
Adding competition features ...
Adding lag features ...
Adding rolling window features ...
Dropped 31,220 rows with NaN lags (first 28 days of each store's history)
Train: 2013-01-29 to 2014-12-31 — 617,140 rows
Test : 2015-01-01 to 2015-07-31 — 196,032 rows

Feature set columns (48):
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Month', 'Year', 'DaysToNextHoliday', 'ABC_Class', 'XYZ_Class', 'CV', 'WeekOfYear', 'Quarter', 'DayOfYear', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'IsGermanHoliday', 'IsPreHoliday', 'Promo2Active', 'HasCompetitor', 'CompOpenYear', 'CompOpenMonth', 'MonthsSinceCompOpen', 'Sales_lag_1', 'Sales_lag_7', 'Sales_lag_14', 'Sales_